# ✂️ KhanCutShorts
Turn any video into Shorts / Reels / TikTok format with smart subject tracking.

Run the cells in order (tap ▶). Turn on a GPU first: Runtime → Change runtime type → T4 GPU.

## 1. Install

In [ ]:
!apt-get -qq install -y ffmpeg
!git clone https://github.com/kamilstanuch/AutoCrop-Vertical.git
!pip install -q -r /content/AutoCrop-Vertical/requirements.txt
!pip install -q -U yt-dlp
%cd /content/AutoCrop-Vertical

## 2. Choose source, ratio and convert
Pick **SOURCE**, fill in only the matching field, choose a **RATIO**, then run.
- **YouTube link**: paste the link in `YOUTUBE_URL`
- **Upload from device**: a file picker appears after you run
- **Google Drive**: put the path in `DRIVE_PATH` (e.g. `MyDrive/videos/clip.mp4`)
- **Ratio**: pick one from the list, or choose `Custom` and type your own in `CUSTOM_RATIO` (e.g. `5:7`)

In [ ]:
#@title KhanCutShorts: Source + Ratio + Convert { display-mode: "form" }
SOURCE = "YouTube link"  #@param ["YouTube link", "Upload from device", "Google Drive"]
YOUTUBE_URL = ""  #@param {type:"string"}
MAX_HEIGHT = "1080"  #@param ["720", "1080", "1440", "2160"]
DRIVE_PATH = "MyDrive/your_video.mp4"  #@param {type:"string"}
RATIO_CHOICE = "9:16 (Shorts / Reels / TikTok)"  #@param ["9:16 (Shorts / Reels / TikTok)", "4:5 (Instagram portrait)", "3:4", "2:3", "1:1 (Square)", "16:9 (Landscape)", "21:9 (Ultrawide)", "Custom"]
CUSTOM_RATIO = "9:16"  #@param {type:"string"}
QUALITY = "balanced"  #@param ["fast", "balanced", "high"]
OUTPUT_NAME = "KhanCutShorts"  #@param {type:"string"}

import os, glob, re

RATIO = CUSTOM_RATIO.strip() if RATIO_CHOICE == "Custom" else RATIO_CHOICE.split(" ")[0]
if not re.fullmatch(r"\d+:\d+", RATIO):
    raise SystemExit(f"Invalid ratio '{RATIO}'. Use the form W:H, e.g. 9:16")
print("Ratio:", RATIO)

for f in glob.glob("/content/input_video.*"):
    os.remove(f)

if SOURCE == "YouTube link":
    url = YOUTUBE_URL.strip()
    if not url:
        raise SystemExit("Paste a YouTube link into YOUTUBE_URL.")
    !yt-dlp -f "bv*[height<={MAX_HEIGHT}]+ba/b[height<={MAX_HEIGHT}]" --merge-output-format mp4 --no-playlist -o "/content/input_video.%(ext)s" "{url}"
    matches = glob.glob("/content/input_video.*")
    if not matches:
        raise SystemExit("Download failed. See the error above.")
    INPUT_PATH = matches[0]

elif SOURCE == "Upload from device":
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file uploaded.")
    INPUT_PATH = os.path.join("/content", list(uploaded.keys())[0])

else:  # Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    INPUT_PATH = os.path.join("/content/drive", DRIVE_PATH.strip().lstrip("/"))

if not os.path.exists(INPUT_PATH):
    raise SystemExit(f"Video not found: {INPUT_PATH}")
print("Using:", INPUT_PATH)

safe_name = re.sub(r"[^A-Za-z0-9_-]", "_", OUTPUT_NAME.strip() or "KhanCutShorts")
OUTPUT = f"/content/{safe_name}.mp4"
%cd /content/AutoCrop-Vertical
!python3 main.py -i "{INPUT_PATH}" -o "{OUTPUT}" --ratio {RATIO} --quality {QUALITY}

## 3. Download

In [ ]:
from google.colab import files
files.download(OUTPUT)
# To save to Drive instead (if Drive is mounted):
# !cp "{OUTPUT}" /content/drive/MyDrive/